# 03 - Entrenamiento Modelo A en SageMaker

Toma el mismo `script.py` que corrimos local en el notebook 01/02 y lo ejecuta como SageMaker Training Job.

**Patron canonico**: slide 02 (estimator + fit + metric_definitions) + slide 04 (registro en Model Registry).

## Pre-requisitos
- Buckets y roles aplicados con `make apply` (Bloque 1).
- Dataset subido a `s3://mlemons-raw-<acct>/lemons/` (script: `aws s3 sync data/lemons s3://.../lemons/`).
- AWS CLI con sesion activa (`aws configure export-credentials` funciona).

In [ ]:
import os
import subprocess
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch

# Cargar credenciales temporales (AWS CLI v2 usa session login en este equipo)
creds = subprocess.check_output(['aws', 'configure', 'export-credentials', '--format', 'env-no-export']).decode()
for line in creds.strip().split('\n'):
    k, _, v = line.partition('=')
    os.environ[k] = v

region = 'us-east-2'
boto_sess = boto3.Session(region_name=region)
sm_session = sagemaker.Session(boto_session=boto_sess)
account = boto_sess.client('sts').get_caller_identity()['Account']
role = f'arn:aws:iam::{account}:role/mlemons-sagemaker-role'

bucket_raw = f'mlemons-raw-{account}'
bucket_models = f'mlemons-models-{account}'
print('Account :', account)
print('Region  :', region)
print('Role    :', role)
print('Raw     :', bucket_raw)
print('Models  :', bucket_models)

## Datos

Dataset ya esta en `s3://mlemons-raw-<acct>/lemons/`. Sanity check de objetos:

In [ ]:
train_s3 = f's3://{bucket_raw}/lemons'
print('Train channel:', train_s3)

s3 = boto_sess.client('s3')
paginator = s3.get_paginator('list_objects_v2')
count = sum(len(p.get('Contents', [])) for p in paginator.paginate(Bucket=bucket_raw, Prefix='lemons/'))
print(f'Objetos en train channel: {count}')

## PyTorch estimator

- `framework_version = '2.0.1'`, `py_version = 'py310'` alineado con el container ECR del slide 05.
- `instance_type = 'ml.t3.medium'` (default account quota).
- `metric_definitions` capturan los prints de `Test accuracy/Test loss` del script.
- `source_dir = '../src/modelo_a'` para que SageMaker incluya `dataset.py` y `transforms.py`.

In [ ]:
estimator = PyTorch(
    entry_point='script.py',
    source_dir='../src/modelo_a',
    role=role,
    framework_version='2.0.1',
    py_version='py310',
    instance_count=1,
    instance_type='ml.t3.medium',
    output_path=f's3://{bucket_models}/modelo-a/',
    base_job_name='mlemons-modelo-a',
    hyperparameters={
        'epochs': 3,
        'batch_size': 32,
        'lr': 1e-3,
    },
    metric_definitions=[
        {'Name': 'test:accuracy', 'Regex': r'Test accuracy: ([0-9.]+)'},
        {'Name': 'test:loss',     'Regex': r'Test loss: ([0-9.]+)'},
    ],
    sagemaker_session=sm_session,
    max_run=3600,
)

In [ ]:
estimator.fit({'train': train_s3}, wait=True)
job_name = estimator.latest_training_job.name
model_data = estimator.model_data
print('Job   :', job_name)
print('Model :', model_data)

## Metricas finales del job

In [ ]:
sm = boto_sess.client('sagemaker')
desc = sm.describe_training_job(TrainingJobName=job_name)
metrics = {m['MetricName']: m['Value'] for m in desc.get('FinalMetricDataList', [])}
print('Final metrics:')
for k, v in metrics.items():
    print(f'  {k:18s} = {v}')

## MLflow tracking (backend local)

Patron del slide 04 con backend file-based en `mlruns/` (gitignored). Para Databricks CE en el futuro: `mlflow.set_tracking_uri('databricks')`.

In [ ]:
import mlflow

mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('MLemons-ModeloA')

with mlflow.start_run(run_name=job_name) as run:
    mlflow.log_param('framework', 'pytorch-2.0.1')
    mlflow.log_param('instance_type', 'ml.t3.medium')
    mlflow.log_param('epochs', 3)
    mlflow.log_param('batch_size', 32)
    mlflow.log_param('lr', 1e-3)
    for k, v in metrics.items():
        mlflow.log_metric(k.replace(':', '_'), v)
    mlflow.log_param('model_data_s3', model_data)
    mlflow.log_param('training_job_name', job_name)
    print('MLflow run:', run.info.run_id)

## Registrar Model Package

Crear `ModelPackageGroup` (idempotente) + Model Package con `PendingManualApproval`. Aprobacion manual antes de promover a endpoint.

In [ ]:
group_name = 'mlemons-modelo-a-group'

try:
    sm.create_model_package_group(
        ModelPackageGroupName=group_name,
        ModelPackageGroupDescription='Modelo A binario (sano vs defectuoso) - clasificacion de limones',
    )
    print('Grupo creado:', group_name)
except sm.exceptions.ResourceInUse:
    print('Grupo ya existe:', group_name)

image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch',
    region=region,
    version='2.0.1',
    py_version='py310',
    instance_type='ml.t3.medium',
    image_scope='inference',
)
print('Inference image:', image_uri)

resp = sm.create_model_package(
    ModelPackageGroupName=group_name,
    ModelPackageDescription=f'Trained job {job_name}',
    InferenceSpecification={
        'Containers': [{
            'Image': image_uri,
            'ModelDataUrl': model_data,
        }],
        'SupportedContentTypes': ['application/x-image'],
        'SupportedResponseMIMETypes': ['application/json'],
    },
    ModelApprovalStatus='PendingManualApproval',
)
package_arn = resp['ModelPackageArn']
print('Model package:', package_arn)

## Aprobar manualmente

Tras revisar las metricas:

In [ ]:
sm.update_model_package(
    ModelPackageArn=package_arn,
    ModelApprovalStatus='Approved',
    ApprovalDescription='Cumple objetivo accuracy >= 0.90',
)
print('Aprobado:', package_arn)

## Salida para el Bloque 3

Bloque 3 (endpoint) consume:
- `MODEL_PACKAGE_ARN`
- `MODEL_DATA_S3`
- `IMAGE_URI`